*Part 3 of 4 · Patterns* · [← index](README.md)

# Patterns

---
## 1. Two pointers

**Complexity:** O(n) time, O(1) space. O(n²) for the sort-and-fix form.

`l, r = 0, len(x) - 1` then `while l < r` — strict, because `l == r` is one element compared
against itself. You must be able to justify why moving a given pointer is safe. The
sort-and-fix form (three-sum) anchors one index and runs the same scan on what remains;
sorting is what makes both the scan and the duplicate-skipping possible.

In [ ]:
def two_sum_sorted(nums, target):
    """Sorted input: the sum tells you unambiguously which end to move."""
    # O(n) time, O(1) space.
    l, r = 0, len(nums) - 1
    while l < r:
        total = nums[l] + nums[r]
        if total == target:
            return [l, r]
        elif total < target:
            l += 1                # need MORE, only the left end can grow it
        else:
            r -= 1                # need LESS, only the right end can shrink it
    return []

In [ ]:
def is_palindrome(s):
    """Alphanumeric only, case-insensitive."""
    # O(n) time, O(1) space.
    l, r = 0, len(s) - 1
    while l < r:
        while l < r and not s[l].isalnum():   # skip junk from both ends
            l += 1
        while l < r and not s[r].isalnum():
            r -= 1
        if s[l].lower() != s[r].lower():
            return False
        l, r = l + 1, r - 1
    return True

In [ ]:
def max_area(heights):
    """Container with most water."""
    # O(n) time, O(1) space.
    l, r, best = 0, len(heights) - 1, 0
    while l < r:
        best = max(best, (r - l) * min(heights[l], heights[r]))
        if heights[l] < heights[r]:
            l += 1                # move the SHORTER wall; the taller one can't help
        else:
            r -= 1
    return best

In [ ]:
def three_sum(nums):
    """All unique triplets summing to zero."""
    # O(n^2) time, O(1) space excluding the output.
    nums.sort()                                   # required for the scan AND the dedup
    res = []
    for i in range(len(nums)):
        if nums[i] > 0:                           # sorted: no way back to zero
            break
        if i > 0 and nums[i] == nums[i - 1]:      # skip duplicate anchors
            continue
        l, r = i + 1, len(nums) - 1
        while l < r:
            total = nums[i] + nums[l] + nums[r]
            if total < 0:
                l += 1
            elif total > 0:
                r -= 1
            else:
                res.append([nums[i], nums[l], nums[r]])
                l += 1
                while l < r and nums[l] == nums[l - 1]:   # skip duplicate seconds
                    l += 1
    return res

---
## 2. Fast and slow pointers

**Complexity:** O(n) time, O(1) space.

Find the middle, detect a cycle, or reach the nth node from the end in a single pass without
ever knowing the length. The guard is `while fast and fast.next` — both checks, because you
are about to dereference two levels deep.

In [ ]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

In [ ]:
def find_middle(head):
    """For even lengths this returns the SECOND middle."""
    # O(n) time, O(1) space.
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

In [ ]:
def has_cycle(head):
    """Floyd's tortoise and hare: in a loop, fast inevitably laps slow."""
    # O(n) time, O(1) space.
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
        if slow is fast:
            return True
    return False

In [ ]:
def remove_nth_from_end(head, n):
    """Gap technique: start fast n ahead, then advance both together."""
    # O(n) time, O(1) space.
    dummy = ListNode(0, head)
    slow = fast = dummy
    for _ in range(n):
        fast = fast.next          # open a gap of n
    while fast.next:
        slow = slow.next          # slow lands just BEFORE the target
        fast = fast.next
    slow.next = slow.next.next
    return dummy.next

---
## 3. Sliding window

**Complexity:** O(n) time, O(1) or O(k) space.

Fixed size: add the new element and drop the old one, never recompute the window. Variable
size: `for r` over the input with a `while` inside that shrinks from `l` until the window is
valid again — a `while`, not an `if`, because one addition can force several removals. It
stays O(n) because each element enters and leaves at most once. *Longest* records after the
shrink loop, *shortest* records inside it. With counts, `del` zero-count keys if your
validity test reads `len(count)`.

In [ ]:
def max_sum_subarray(nums, k):
    """Fixed window: best sum over every subarray of length exactly k."""
    # O(n) time, O(1) space.
    if len(nums) < k:
        return None
    window = sum(nums[:k])            # first window, computed once
    best = window
    for r in range(k, len(nums)):
        window += nums[r]             # gain the new element
        window -= nums[r - k]         # lose the old one
        best = max(best, window)
    return best

In [ ]:
def longest_ones(nums, k):
    """Variable window: longest run of 1s if you may flip at most k zeros."""
    # O(n) time, O(1) space.
    l = zeros = best = 0
    for r in range(len(nums)):
        if nums[r] == 0:
            zeros += 1
        while zeros > k:              # WHILE, not if
            if nums[l] == 0:
                zeros -= 1
            l += 1
        best = max(best, r - l + 1)   # longest: record AFTER shrinking
    return best

In [ ]:
def min_subarray_len(nums, target):
    """Shortest subarray summing to >= target."""
    # O(n) time, O(1) space.
    l = total = 0
    best = float("inf")
    for r in range(len(nums)):
        total += nums[r]
        while total >= target:        # while still valid, try to shrink
            best = min(best, r - l + 1)   # shortest: record INSIDE
            total -= nums[l]
            l += 1
    return best if best != float("inf") else 0

In [ ]:
def longest_unique_substring(s):
    """Longest substring with no repeated character."""
    # O(n) time, O(k) space.
    seen = {}                          # char -> most recent index
    l = best = 0
    for r, ch in enumerate(s):
        if ch in seen and seen[ch] >= l:
            l = seen[ch] + 1           # jump past the previous occurrence
        seen[ch] = r
        best = max(best, r - l + 1)
    return best

In [ ]:
from collections import Counter


def character_replacement(s, k):
    """Longest same-character run if you may replace k characters."""
    # O(n) time, O(k) space.
    count = Counter()
    l = best = max_freq = 0
    for r in range(len(s)):
        count[s[r]] += 1
        max_freq = max(max_freq, count[s[r]])
        while (r - l + 1) - max_freq > k:      # too many chars left to replace
            count[s[l]] -= 1
            if count[s[l]] == 0:
                del count[s[l]]                # keep len(count) honest
            l += 1
        best = max(best, r - l + 1)
    return best

---
## 4. Monotonic stack

**Complexity:** O(n) time, O(n) space — every index is pushed once and popped at most once.

`for i, v in enumerate(nums)`, then `while stack and <top loses to v>`: pop and resolve it,
finally push `i`. Store **indices**, not values, when the answer needs a distance. The stack
stays sorted by construction, and that invariant *is* the pattern. Whatever is still on the
stack at the end never found its match.

In [ ]:
def daily_temperatures(temps):
    """For each day, how many days until a warmer one. 0 if never."""
    # O(n) time, O(n) space.
    res = [0] * len(temps)
    stack = []                         # indices, temps DECREASING bottom to top
    for i, t in enumerate(temps):
        while stack and temps[stack[-1]] < t:
            j = stack.pop()            # day j finally found its warmer day
            res[j] = i - j             # indices are why we can measure the gap
        stack.append(i)
    return res                         # anything left keeps its 0

In [ ]:
def next_greater(nums):
    """-1 where no greater element exists to the right."""
    # O(n) time, O(n) space.
    res = [-1] * len(nums)
    stack = []
    for i, v in enumerate(nums):
        while stack and nums[stack[-1]] < v:
            res[stack.pop()] = v
        stack.append(i)
    return res

---
## 5. Prefix and suffix

**Complexity:** O(n) time, O(1) space excluding the output.

One pass left to right building running values, one pass right to left folding them in. The
prefix at index `i` must *exclude* `nums[i]`, so write into `res[i]` before folding `nums[i]`
into the running total — the order of those two lines is the whole trick. Prefix sums also
turn repeated range queries into O(1) lookups.

In [ ]:
def product_except_self(nums):
    # O(n) time, O(1) space excluding the output.
    n = len(nums)
    res = [1] * n

    prefix = 1
    for i in range(n):
        res[i] = prefix            # write BEFORE including nums[i]
        prefix *= nums[i]

    suffix = 1
    for i in range(n - 1, -1, -1):
        res[i] *= suffix           # fold the right-hand side in
        suffix *= nums[i]

    return res

In [ ]:
def prefix_sums(nums):
    """pre[i] = sum of nums[:i]. Range sum [i, j) is then pre[j] - pre[i], in O(1)."""
    # O(n) time, O(n) space.
    pre = [0]
    for v in nums:
        pre.append(pre[-1] + v)
    return pre

---
## 6. Hash set membership

**Complexity:** O(n) time, O(n) space — the classic trade.

O(1) lookups replace a nested loop: duplicates, complements, "does this neighbour exist?".
In longest-consecutive, only start counting from a number whose predecessor is *absent* —
that guard is what keeps it O(n) rather than re-walking every sequence from every position.

In [ ]:
def longest_consecutive(nums):
    # O(n) time, O(n) space.
    num_set = set(nums)
    best = 0
    for n in num_set:
        if n - 1 in num_set:
            continue               # not a sequence START -- someone else counts it
        length = 1
        while n + length in num_set:
            length += 1
        best = max(best, length)
    return best

In [ ]:
def two_sum(nums, target):
    """Unsorted: store what you have seen, look up the complement."""
    # O(n) time, O(n) space.
    seen = {}                      # value -> index
    for i, v in enumerate(nums):
        if target - v in seen:
            return [seen[target - v], i]
        seen[v] = i
    return []

---
## 7. Tree recursion

**Complexity:** O(n) time, O(h) space.

Three variants, and every non-trivial tree problem is one of them: return a value up from
each subtree, thread state *down* through the arguments, or record into an outer variable
with `nonlocal`. Diameter is the trap — the function *returns* height but *records*
diameter, because the best path may not pass through the root and so cannot be the return
value.

In [ ]:
from collections import deque


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(vals):
    """LeetCode level-order format, None for missing. [1,2,3,None,4] -> root."""
    # O(n) time, O(n) space.
    if not vals:
        return None
    root = TreeNode(vals[0])
    queue = deque([root])
    i = 1
    while queue and i < len(vals):
        node = queue.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i])
            queue.append(node.left)   # only real nodes get enqueued
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i])
            queue.append(node.right)
        i += 1
    return root

In [ ]:
def diameter(root):
    """Longest path between any two nodes. Returns height, RECORDS diameter."""
    # O(n) time, O(h) space.
    best = 0

    def height(node):
        nonlocal best
        if not node:
            return 0
        l = height(node.left)
        r = height(node.right)
        best = max(best, l + r)        # path THROUGH this node -- recorded, not returned
        return 1 + max(l, r)           # what the parent actually needs

    height(root)
    return best

In [ ]:
def has_path_sum(root, target):
    """State passed DOWN: each call gets the remaining budget."""
    # O(n) time, O(h) space.
    if not root:
        return False
    remaining = target - root.val
    if not root.left and not root.right:       # leaf: the decision point
        return remaining == 0
    return has_path_sum(root.left, remaining) or has_path_sum(root.right, remaining)